# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd

Cloning into 'FlyRank-ML'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (318/318), done.
remote: Compressing objects: 100% (265/265), done.
remote: Total 318 (delta 190), reused 104 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (318/318), 2.16 MiB | 15.36 MiB/s, done.
Resolving deltas: 100% (190/190), done.


In [2]:
df = pd.read_csv('/content/FlyRank-ML/data/raw/content_refresh_anonymized.csv')

## 1. Two paper findings + my methodology questions

I picked both findings from the ML appendix, since they run the closest task to my own Lane 2 work
(predicting decline/growth from portfolio signals) and I can check them against habits I already
had to build in w05.

---

### Finding A — "What Predicts Health?" (Random Forest feature importance)

**Claim:** Random Forest feature importance for predicting `health_score`. Top features: Average
Position (43%), Impressions (32%), Scroll Depth (15%).

**Where does the label come from?**
`health_score` is defined in the paper's own Methodology section as:
`Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)`.

**Does the validation design carry the claim?**
No — and the paper partly knows it. Three of the four top features (Position, Impressions, Scroll
Depth) are direct ingredients of the label itself. This fails gate 1 of the leakage rule I used in
w05: *exclude a feature if it was used to compute the label*. A holdout split doesn't rescue this,
because the leakage isn't about train/test contamination — it's that the "prediction" is
arithmetically close to the target by construction. The paper does add a caveat ("importance is
descriptive rather than causal"), which is honest, but the section title — "What Predicts
Health?" — still implies a discovery. A model recovering its own formula isn't a discovery, it's
a confirmation that the arithmetic works.

**My question:** what would this feature-importance chart look like if `health_score`'s four
input components were excluded and only independent signals (word count, age, AI sessions,
content type) were used? That's the real "what predicts health" question — this version doesn't
answer it.

---

### Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

**Claim:** Logistic regression separating growing vs. declining pages, reported at 71% holdout
accuracy, using an 80/20 split.

**Where does the label come from?**
Growth/decline is derived from 30-day-vs-previous-30-day impression change (per the paper's own
Trend Direction definition: Up >10%, Down >10%).

**Does the validation design carry the claim?**
Two gaps:

1. **No base rate reported.** 71% accuracy is only meaningful next to the majority-class rate.
   If growing pages are, say, 62% of the sample, 71% is 9 points of real skill — not 71. My w05
   notebook made this the same way: comparing model output to a `DummyClassifier` baseline before
   trusting any accuracy number. The paper reports the metric with no baseline beside it.
2. **Plain 80/20 split across 57 brands, not grouped.** The Methodology section confirms: "Random
   Forest (80/20 split), Logistic Regression (80/20 split)" — no mention of grouping by brand.
   This is the exact vulnerability I found empirically in w05: per-client `down` rates in my own
   data ranged from ~0% to ~94%. If FlyRank's 57 brands have similarly uneven growth/decline
   rates, a random split lets rows from the same brand land in both train and test, and the model
   can partly learn "brand style" instead of a generalizable growth signal. `StratifiedGroupKFold`
   grouped by brand (the equivalent of my `client_id` grouping) would be the honest test here.

**My question:** would 71% holdout accuracy survive a brand-grouped split, and what is the
majority-class baseline it should be compared against?

---

Both findings are framed carefully in the prose ("descriptive rather than causal," "exploratory
appendix," "do not override direct portfolio evidence"), so this isn't a case of overclaiming in
the text. The gap is in the validation design underneath the numbers, not the wording around them.

## 2. My model under an honest split (before/after)

Same model (`RandomForestClassifier`, `min_samples_leaf=20`), same safe feature list, same K
values — the only thing that changed between the two runs is the split design.

**Before (naive random split):** 31 of 32 clients appear in *both* train and test. With 30,000
rows spread across 32 clients, a random 80/20 split was never going to keep clients cleanly
separated — nearly every client has enough rows to land pages on both sides by chance alone.

**After (honest grouped split, `StratifiedGroupKFold` by `client_id`):** 0 clients appear in
both. Every page from a given client sits entirely on one side, which is the actual guarantee
this split design is supposed to provide.

| K | Base rate | Precision@K (random split) | Precision@K (grouped split) | Gap |
|---|---|---|---|---|
| 20 | 0.542 | 0.900 | 0.800 | 0.100 |
| 50 | 0.542 | 0.960 | 0.720 | 0.240 |
| 100 | 0.542 | 0.950 | 0.690 | 0.260 |
| 500 | 0.542 | 0.902 | 0.686 | 0.216 |
| 1000 | 0.542 | 0.857 | 0.671 | 0.186 |

**The random split is inflated at every K**, and the inflation is worst exactly where it matters
most for a real review queue — K=50 and K=100, a 0.24–0.26 point gap. This isn't noise; it's the
model quietly using client identity as a shortcut. With 31 of 32 clients present on both sides,
the model can partly learn "what does *this specific client's* declining page usually look like"
during training, then get graded on more pages from that same client at test time. That's an
easier task than the one the deliverable is actually supposed to solve — ranking pages it has
never effectively seen anything from that client about.

**The honest number is still real skill, just smaller.** At K=1000, grouped-split precision is
0.671 against a 0.542 base rate — genuine signal above the floor, not nothing. The random split
wasn't measuring a *different* model; it was measuring the same model's performance under an
easier, less honest test.

**This matches, and sharpens, what w05 already found.** The w05 calibration check showed the
random split "looked" nearly perfectly calibrated while the grouped split exposed a real
miscalibration problem — I read that at the time as the random split hiding a defect. This table
shows the same mechanism from the other direction: the random split doesn't just hide problems,
it actively manufactures precision that the honest split can't reproduce. The grouped split isn't
being conservative or pessimistic here — it's the only one of the two numbers that would survive
contact with a genuinely new client.

In [3]:
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# --- Rebuild the w05 feature set and label (same safe list, ML-04/ML-05 locked) ---
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

safe_numeric = [
    'content_age_days', 'days_since_last_update', 'word_count', 'char_count',
    'search_volume', 'competition', 'cpc',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'
]
safe_categorical = [
    'content_type', 'main_intent', 'competition_level',
    'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier'
]

for col in safe_numeric:
    df[col] = df[col].fillna(0)
for col in safe_categorical:
    df[col] = df[col].fillna('unknown')

X = df[safe_numeric + safe_categorical].copy()
y = df['is_declining_label']

preprocessor = ColumnTransformer([
    ('num', 'passthrough', safe_numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), safe_categorical)
])

def make_pruned_rf():
    return Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                         random_state=42, class_weight='balanced'))
    ])

def precision_at_k(df_, score_col, label_col, k):
    top_k = df_.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

# --- BEFORE: naive random split (stratified on label, NOT grouped by client) ---
X_train_rand, X_test_rand, y_train_rand, y_test_rand, idx_train_rand, idx_test_rand = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)
model_rand = make_pruned_rf()
model_rand.fit(X_train_rand, y_train_rand)

test_rand_df = df.loc[idx_test_rand].copy()
test_rand_df['proba'] = model_rand.predict_proba(X_test_rand)[:, 1]

# client leakage check — this is what the random split allows and the grouped split forbids
clients_in_both = set(df.loc[idx_train_rand, 'client_id']) & set(df.loc[idx_test_rand, 'client_id'])
print(f"[Random split] Clients appearing in BOTH train and test: {len(clients_in_both)} "
      f"of {df['client_id'].nunique()} total clients")

# --- AFTER: honest grouped split (StratifiedGroupKFold by client_id, same as w05) ---
model_df = df[['client_id'] + safe_numeric + safe_categorical + ['is_declining_label']].copy()
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(model_df, model_df['is_declining_label'], groups=model_df['client_id']))

X_train_grp, y_train_grp = X.iloc[train_idx], y.iloc[train_idx]
X_test_grp, y_test_grp = X.iloc[test_idx], y.iloc[test_idx]

model_grp = make_pruned_rf()
model_grp.fit(X_train_grp, y_train_grp)

test_grp_df = df.iloc[test_idx].copy()
test_grp_df['proba'] = model_grp.predict_proba(X_test_grp)[:, 1]

clients_grp_both = set(df.iloc[train_idx]['client_id']) & set(df.iloc[test_idx]['client_id'])
print(f"[Grouped split]  Clients appearing in BOTH train and test: {len(clients_grp_both)} "
      f"of {df['client_id'].nunique()} total clients")

# --- Comparison table: same model, same K values, only the split differs ---
base_rate = y.mean()
print(f"\nBase rate (overall down-rate): {base_rate:.3f}\n")

rows = []
for k in [20, 50, 100, 500, 1000]:
    rows.append({
        'K': k,
        'base_rate': round(base_rate, 3),
        'precision@K_random_split': round(precision_at_k(test_rand_df, 'proba', 'is_declining_label', k), 3),
        'precision@K_grouped_split': round(precision_at_k(test_grp_df, 'proba', 'is_declining_label', k), 3),
    })
comparison = pd.DataFrame(rows)
comparison['gap'] = (comparison['precision@K_random_split'] - comparison['precision@K_grouped_split']).round(3)
print(comparison)

[Random split] Clients appearing in BOTH train and test: 31 of 32 total clients
[Grouped split]  Clients appearing in BOTH train and test: 0 of 32 total clients

Base rate (overall down-rate): 0.542

      K  base_rate  precision@K_random_split  precision@K_grouped_split    gap
0    20      0.542                     0.900                      0.800  0.100
1    50      0.542                     0.960                      0.720  0.240
2   100      0.542                     0.950                      0.690  0.260
3   500      0.542                     0.902                      0.686  0.216
4  1000      0.542                     0.857                      0.671  0.186


## 3. Leakage audit

Re-ran the w03 hunt against the actual feature set shipped in w05/w06, not just the feature set
as originally planned — the point of "the same hunt on your final feature set" is to catch drift
between what got audited and what got used.

**Confirmed-leaky columns: none present.** The full `confirmed_leaky` set from w03 (all `_90d`
totals, `_last_30d` columns, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`,
`ai_traffic_pct`, `impression_tier`, `position_tier`, `days_with_impressions`,
`days_with_sessions`, `trend_direction`, `trend_pct`) has zero overlap with `safe_numeric` +
`safe_categorical`. The exclusion held from w03 through to the model actually trained in w05.

**Both structural tests re-verified at the original strength.** `prev_30d + last_30d` still fits
inside `90d` for 100% of rows — the 90-day totals still structurally contain the label's future
window, confirming they're correctly excluded. `trend_pct`'s correlation with computed
impressions percent-change is still 1.000 — `impressions_last_30d` is still the label's literal
source and is still correctly excluded. Nothing drifted between w03 and now.

**A real catch: `competition_level` should have been dropped, and wasn't.** w03's own conclusion
(Section 4) states `competition_level` is redundant with `competition` and should be dropped —
verified there with non-overlapping bins. Re-checking that here confirms the same thing: `LOW` =
0.00–0.33, `MEDIUM` = 0.33–0.66, `HIGH` = 0.67–1.00, `unknown` = 0.00 (fill value) — clean,
deterministic binning of `competition`, no ambiguity. But `competition_level` is still sitting in
`safe_categorical` in both w05's model and the Section 2 code above. My own written audit and my
own shipped code disagree with each other.

**This is not leakage — it's worth being precise about that distinction.** `competition` and
`competition_level` were both measured before prediction time and neither was used to build
`is_declining_label`; they pass both gates of the leakage rule. Keeping both doesn't let the model
see the future or see the answer — it just means the model gets the same information twice, once
as a number and once as a redundant one-hot-encoded category. That inflates the categorical
feature count for no signal gain, but it doesn't invalidate the Precision@K numbers reported in
Section 2 the way a real leak would.

**Fix, not urgent:** drop `competition_level` from `safe_categorical` before the capstone pass.
Leaving it in for this audit doesn't change any conclusion already drawn — the model's
Precision@K numbers in Section 2 are unaffected by a redundant feature, only slightly less
efficient. Flagging it here instead of silently fixing it, since the assignment is about catching
exactly this kind of gap between what was decided and what actually shipped.

In [4]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# --- Final feature set, as actually used in w05 / w06 Section 2 ---
safe_numeric = [
    'content_age_days', 'days_since_last_update', 'word_count', 'char_count',
    'search_volume', 'competition', 'cpc',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'
]
safe_categorical = [
    'content_type', 'main_intent', 'competition_level',
    'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier'
]
final_features = safe_numeric + safe_categorical

# --- Check 1: re-run the w05 assert — no confirmed-leaky column snuck in ---
confirmed_leaky = {
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'impression_tier', 'position_tier', 'days_with_impressions', 'days_with_sessions',
    'trend_direction', 'trend_pct'
}
overlap = confirmed_leaky & set(final_features)
print("Confirmed-leaky columns present in final feature set:", overlap if overlap else "NONE — clean")

# --- Check 2: re-run the _90d nesting test (w03 Section 3, re-verified here) ---
check = df[['impressions_prev_30d', 'impressions_last_30d', 'impressions_90d']].dropna()
nested = (check['impressions_prev_30d'] + check['impressions_last_30d']) <= check['impressions_90d']
print(f"\n_90d nesting test (re-verified): {nested.mean():.3f} of rows show "
      f"prev_30d + last_30d fitting inside 90d")

# --- Check 3: re-run the trend_pct source test (w03 Section 3, re-verified here) ---
sample = df[['impressions_prev_30d', 'impressions_last_30d', 'trend_pct']].dropna()
sample = sample[sample['impressions_prev_30d'] > 0]
sample['computed_pct'] = ((sample['impressions_last_30d'] - sample['impressions_prev_30d'])
                           / sample['impressions_prev_30d']) * 100
corr = sample[['trend_pct', 'computed_pct']].corr().iloc[0, 1]
print(f"trend_pct source test (re-verified): corr with computed impressions %% change = {corr:.3f}")

# --- Check 4: competition_level vs competition — is it actually redundant? ---
# w03 concluded this should be DROPPED. It is still present in safe_categorical above.
print(f"\ncompetition_level currently in final feature set: "
      f"{'competition_level' in final_features}")
print(df.groupby('competition_level')['competition'].describe()[['min', 'max']])

Confirmed-leaky columns present in final feature set: NONE — clean

_90d nesting test (re-verified): 1.000 of rows show prev_30d + last_30d fitting inside 90d
trend_pct source test (re-verified): corr with computed impressions %% change = 1.000

competition_level currently in final feature set: True
                    min   max
competition_level            
HIGH               0.67  1.00
LOW                0.00  0.33
MEDIUM             0.33  0.66
unknown            0.00  0.00


## 4. Claim rewrite

**Original (w04 / ML-06, Signal Test #1):**

> "gpt-4o-mini is the strongest performer (0.34% weighted CTR), followed closely by
> gemini-3-flash-preview (0.32%) and gemini-2.5-flash (0.32%), with gpt-5-mini clearly the
> weakest (0.26%)."

**What's wrong with it, on the claim ladder:**

"Strongest performer" and "clearly the weakest" read as a capability ranking — as if some models
write inherently better content, in general, and would keep winning on a different dataset or
different client mix. What I actually have is a single measured comparison, in one snapshot, of
weighted CTR grouped by which model generated the content. That's the "measured comparison
between groups" tier of the claim ladder, not a settled ranking. The honest words are "showed" or
"associated with," not "is the strongest."

**The confound I hadn't named yet:** which model wrote a page probably isn't random. If certain
clients or content types were more likely to use one model over another during the period this
data covers, the CTR difference could be tracking *client* or *topic*, not *model quality* — the
same client-concentration issue from ML-02 (a handful of clients hold ~76% of rows) could easily
be riding along inside this comparison without me having checked for it. I didn't control for
that in the original write-up, and a 0.07 percentage-point gap is modest enough that a real
confound could account for some or all of it.

**Rewrite:**

> "In this portfolio snapshot, pages generated by gpt-4o-mini showed the highest weighted CTR
> (0.34%), and pages generated by gpt-5-mini showed the lowest (0.26%) — a modest 0.07-point
> gap. This is a model-level pattern, not a provider-level one: grouping by provider (openai vs.
> google) would hide which specific model is actually associated with the difference. This
> comparison hasn't been checked against which clients or content types used which model, so
> part of the gap could reflect client or topic mix rather than the model itself. Decision-support
> reading: this is worth a closer look before treating any model as a default choice, not
> evidence that one model produces better-performing content in general."

This keeps the actual number (0.34% vs 0.26%, 0.07-point gap) and the model-vs-provider
correction I'd already made, but removes the "strongest/weakest performer" framing and adds the
confound I'd missed the first time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.